# Modelos Fundacionales Multimodales: Vision + Lenguaje

Los modelos multimodales combinan vision (imagenes) y lenguaje (texto) para entender el mundo de forma mas completa. En este notebook veremos 4 modelos clave:

- **CLIP**: Relaciona imagenes con texto (clasificacion zero-shot)
- **BLIP**: Genera descripciones de imagenes
- **Grounding DINO**: Detecta objetos usando texto natural
- **Qwen2.5-VL**: Razonamiento visual avanzado (VLM)

La ventaja: no necesitas entrenar un modelo nuevo para cada tarea, estos modelos ya entienden conceptos generales.

## Configuracion e Imports

Instalamos dependencias necesarias (solo para Qwen2.5-VL).

In [ ]:
import torch
import transformers
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
import json
import re
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

## Carga de Modelos

Cargamos el modelo desde HF. Esto puede tardar unos minutos la primera vez, dependiendo del tamaño del modelo.

In [ ]:
# CLIP
from transformers import CLIPProcessor, CLIPModel

clip_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model = CLIPModel.from_pretrained(clip_name).to(device)
print(f"✓ CLIP cargado")

## Carga de Imagenes de Ejemplo

Usamos imagenes del repositorio de GitHub para los ejemplos.

In [ ]:
# Cargar imagenes desde GitHub
url_cars = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/cars.jpg"
image_cars = Image.open(BytesIO(requests.get(url_cars).content)).convert("RGB")

url_person_dog = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/person_dog.jpg"
image_person_dog = Image.open(BytesIO(requests.get(url_person_dog).content)).convert("RGB")

url_person_cars = "https://github.com/sergiovillanueva/Modelos_Fundacionales/raw/main/assets/person_cars.jpg"
image_person_cars = Image.open(BytesIO(requests.get(url_person_cars).content)).convert("RGB")

url_dog = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog.jpg"
image_dog = Image.open(BytesIO(requests.get(url_dog).content)).convert("RGB")

url_dog2 = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog2.jpg"
image_dog2 = Image.open(BytesIO(requests.get(url_dog2).content)).convert("RGB")

url_fruits = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/fruits.jpg"
image_fruits = Image.open(BytesIO(requests.get(url_fruits).content)).convert("RGB")

url_bananas = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/bananas.jpg"
image_bananas = Image.open(BytesIO(requests.get(url_bananas).content)).convert("RGB")

url_board = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/board.jpg"
image_board = Image.open(BytesIO(requests.get(url_board).content)).convert("RGB")

url_carpet_ok = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/carpet_ok.jpg"
image_carpet_ok = Image.open(BytesIO(requests.get(url_carpet_ok).content)).convert("RGB")

url_carpet_nok = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/carpet_nok.jpg"
image_carpet_nok = Image.open(BytesIO(requests.get(url_carpet_nok).content)).convert("RGB")

print("Imagenes cargadas")

## CLIP: Clasificacion Zero-Shot

CLIP aprende a relacionar imagenes con texto. Podemos clasificar imagenes sin entrenar, simplemente describiendo las categorias.

**Como funciona:**
1. Le damos una imagen y varias descripciones de texto
2. CLIP calcula la similitud entre la imagen y cada texto
3. Nos dice cual descripcion es mas probable

**Paper**: https://arxiv.org/pdf/2103.00020

In [ ]:
def classify_with_clip(image, text_options):
    """Clasifica imagen usando CLIP con las opciones de texto dadas"""
    
    # Procesar imagen y textos
    inputs = clip_processor(text=text_options, images=image, return_tensors="pt", padding=True).to(device)
    
    # Calcular similitudes
    with torch.no_grad():
        outputs = clip_model(**inputs)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]
    
    # Mostrar resultados
    print("Probabilidades:\n")
    for text, prob in zip(text_options, probs):
        print(f"  {text}: {prob.item()*100:.2f}%")
    
    best_idx = probs.argmax().item()
    best_match = text_options[best_idx]
    
    # Visualizar
    plt.imshow(image)
    plt.title(f"Mejor match: {best_match} ({probs[best_idx].item()*100:.1f}%)")
    plt.axis("off")
    plt.show()
    
    return best_match

# Ejemplo: clasificar tipo de escena
# classify_with_clip(image_cars, ["parking lot with cars", "inside a factory", "office space", "outdoor park"])
# classify_with_clip(image_fruits, ["fruits", "cars", "dogs", "cats"])
# classify_with_clip(image_fruits, ["kiwi", "orange", "lemon"])

## BLIP: Generacion de Descripciones

BLIP genera descripciones automaticas de imagenes en lenguaje natural. Es como tener un asistente que mira la imagen y te dice que ve.

**Uso tipico**: Generar metadatos automaticos, ayudar a personas con discapacidad visual, indexar imagenes.

**Paper**: https://arxiv.org/pdf/2201.12086

In [ ]:
# BLIP
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_name = "Salesforce/blip-image-captioning-base"
blip_processor = BlipProcessor.from_pretrained(blip_name)
blip_model = BlipForConditionalGeneration.from_pretrained(blip_name, torch_dtype=torch.float16).to(device)
print(f"✓ BLIP cargado")

In [ ]:
def generate_caption(image):
    """Genera descripcion de la imagen usando BLIP"""
    
    inputs = blip_processor(image, return_tensors="pt").to(device, torch.float16)
    
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=50)
    
    caption = blip_processor.decode(out[0], skip_special_tokens=True)
    print(f"Descripcion: {caption}")
    
    # Visualizar
    plt.imshow(image)
    plt.title(caption)
    plt.axis("off")
    plt.show()
    
    return caption

# Ejemplo
generate_caption(image_person_cars)
# generate_caption(image_board)
# generate_caption(image_carpet_nok)

## Grounding DINO: Deteccion Guiada por Texto

Grounding DINO detecta objetos usando lenguaje natural. En lugar de categorias fijas (como COCO), puedes pedirle que encuentre "la persona con chaqueta roja" o "todas las tuercas oxidadas".

**Ventaja**: No necesitas reentrenar para nuevas categorias, simplemente cambia el texto.

**Paper**: https://arxiv.org/pdf/2303.05499

In [ ]:
# Grounding DINO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

dino_name = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(dino_name)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_name).to(device)
print(f"✓ Grounding DINO cargado")

In [ ]:
def detect_objects(image, prompt, box_threshold=0.35, text_threshold=0.25):
    """Detecta objetos usando Grounding DINO con prompt de texto"""
    
    w, h = image.size
    
    # Procesar imagen con DINO
    inputs = dino_processor(images=image, text=prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = dino_model(**inputs)
    
    # Post-procesar detecciones
    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[(h, w)]
    )
    
    boxes = results[0]['boxes'].cpu().numpy()
    labels = results[0]['labels']
    scores = results[0]['scores'].cpu().numpy()
    
    print(f"Detectados {len(boxes)} objetos con prompt: '{prompt}'\n")
    
    # Dibujar detecciones
    img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    for box, label, score in zip(boxes, labels, scores):
        x0, y0, x1, y1 = map(int, box)
        cv2.rectangle(img_cv, (x0, y0), (x1, y1), (255, 0, 0), 3)  # Azul
        text = f"{label} {score:.2f}"
        cv2.putText(img_cv, text, (x0, y0-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        print(f"  - {label}: {score:.2f}")
    
    # Visualizar
    plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    plt.title(f"Detecciones: {prompt}")
    plt.axis("off")
    plt.show()
    
    return boxes, labels, scores

# Ejemplo:
detect_objects(image_fruits, "kiwi. apple.")
# detect_objects(image_person_cars, ["yellow car"])
# detect_objects(image_cars, ["yellow car"])

# detect_objects(image_carpet_nok, ["defect"])



## Qwen2.5-VL: Razonamiento Visual Avanzado

Qwen2.5-VL es un Vision-Language Model (VLM) que puede:
- Responder preguntas sobre imagenes
- Razonar sobre contenido visual
- Generar respuestas estructuradas (JSON)
- Detectar objetos con instrucciones complejas

Es como tener un asistente que realmente "entiende" la imagen.

**Paper**: https://arxiv.org/pdf/2502.13923

In [ ]:
try:
    import qwen_vl_utils
except ImportError:
    print("Installing qwen-vl-utils...")
    %pip install qwen-vl-utils -q
    import qwen_vl_utils
    
from transformers import Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# Qwen2.5-VL
qwen_name = "Qwen/Qwen2.5-VL-3B-Instruct"
qwen_processor = AutoProcessor.from_pretrained(qwen_name)
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(qwen_name, torch_dtype=torch.float16, device_map="auto")
print(f"✓ Qwen2.5-VL cargado")

In [ ]:
def ask_qwen(image, question):
    """Pregunta a Qwen2.5-VL sobre una imagen"""
    
    # Crear conversacion
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]}
    ]
    
    # Procesar
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(device)
    
    # Generar respuesta
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=256)
    
    answer = qwen_processor.batch_decode(
        [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)],
        skip_special_tokens=True
    )[0]
    
    print(f"Pregunta: {question}")
    print(f"Respuesta: {answer}\n")
    
    # Visualizar
    plt.imshow(image)
    plt.title(f"Q: {question[:50]}..." if len(question) > 50 else f"Q: {question}")
    plt.axis("off")
    plt.show()
    
    return answer

# Ejemplo: preguntar sobre la imagen
ask_qwen(image_person_cars, "What is the woman doing?")
# ask_qwen(image_person_cars, "Where is the red car?")
# ask_qwen(image_person_cars, "How many cars are in the image?")
# ask_qwen(image_fruits, "How many fruits are in the image?.")
# ask_qwen(image_carpet_nok, "is there any defect in the carpet?")

### Qwen2.5-VL con Deteccion (JSON)

Podemos pedirle a Qwen que detecte objetos y nos devuelva las coordenadas en formato JSON estructurado.

In [ ]:
def detect_with_qwen(image, instruction):
    """Detecta objetos con Qwen2.5-VL y devuelve bounding boxes"""
    
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": instruction}
        ]}
    ]
    
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=256)
    
    answer = qwen_processor.batch_decode(
        [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)],
        skip_special_tokens=True
    )[0]
    
    print(f"Instruccion: {instruction}")
    print(f"Respuesta: {answer}\n")
    
    # Extraer JSON de la respuesta
    json_match = re.search(r'```json\n(.*?)\n```', answer, re.DOTALL)
    
    img_array = np.array(image)
    
    if json_match:
        data = json.loads(json_match.group(1))
        
        # Dibujar bounding boxes
        for item in data:
            if 'bbox_2d' in item:
                x1, y1, x2, y2 = item['bbox_2d']
                label = item.get('label', 'object')
                
                cv2.rectangle(img_array, (x1, y1), (x2, y2), (255, 0, 0), 3)  # Azul
                cv2.putText(img_array, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    # Visualizar
    plt.imshow(img_array)
    plt.title("Detecciones con Qwen2.5-VL")
    plt.axis("off")
    plt.show()
    
    return answer

# Ejemplo: detectar frutas

detect_with_qwen(image_fruits, "Detect fruits and provide bounding box coordinates and labels.")
detect_with_qwen(image_carpet_nok, "Is there any surface defect in the image? Detect the defects and provide bounding box coordinates.")
detect_with_qwen(image_fruits, "Detect the kiwis and provide bounding box coordinates in JSON format.")
detect_with_qwen(image_fruits, "Detect the fruits from New Zealand and provide bounding box coordinates in JSON format.")
detect_with_qwen(image_fruits, "Detect the fruits from Valencia and provide bounding box coordinates in JSON format.")


## Comparativa: Cuando usar cada modelo?

| Modelo | Mejor para | Ventajas | Desventajas |
|--------|-----------|----------|-------------|
| **CLIP** | Clasificacion zero-shot | Rapido, simple | Solo clasificacion |
| **BLIP** | Generar descripciones | Captions naturales | No detecta objetos |
| **Grounding DINO** | Deteccion por texto | Preciso en deteccion | Solo bounding boxes |
| **Qwen2.5-VL** | Razonamiento complejo | Muy flexible, entiende contexto | Mas lento, mas pesado |

**Recomendacion general:**
- Si solo necesitas clasificar → **CLIP**
- Si necesitas descripciones → **BLIP**
- Si necesitas deteccion rapida → **Grounding DINO**
- Si necesitas razonar o tareas complejas → **Qwen2.5-VL**

## Ejercicio: Prueba con tus propias imagenes

Elige una de las funciones que hemos creado y pruebala con diferentes imagenes y prompts.

**Algunas ideas:**
1. Clasifica la imagen con CLIP usando categorias diferentes
2. Genera captions de todas las imagenes con BLIP
3. Usa Grounding DINO para detectar
4. Preguntale a Qwen

In [33]:

# Tu codigo aqui...


## Resumen

En este notebook hemos visto:

✅ Que son los modelos fundacionales multimodales  
✅ **CLIP**: Clasificacion zero-shot relacionando imagen-texto  
✅ **BLIP**: Generacion automatica de descripciones  
✅ **Grounding DINO**: Deteccion de objetos con prompts  
✅ **Qwen2.5-VL**: Razonamiento visual avanzado  
✅ Cuando usar cada modelo segun la tarea  